<a href="https://colab.research.google.com/github/lephuongvy1702/Deep_Learning_Group/blob/gaussian_mlp_project/02_network_structure_and_exp1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BÀI TẬP LỚN: XẤP XỈ HÀM SỐ DÙNG MẠNG MLP BẰNG PYTORCH

## NETWORK STRUCTURE

In [ ]:
class RegressionMLP(nn.Module):
    def __init__(self, H=10, C=1): # Thêm H và C vào đây để nhận tham số truyền vào
        super(RegressionMLP, self).__init__()

        layers = []
        # Tầng ẩn đầu tiên từ input (2) sang H
        layers.append(nn.Linear(2, H))
        layers.append(nn.Sigmoid())

        # Các tầng ẩn tiếp theo nếu C > 1 (đáp ứng Nhóm C - Độ sâu)
        for _ in range(C - 1):
            layers.append(nn.Linear(H, H))
            layers.append(nn.Sigmoid())

        # Tầng đầu ra (Hồi quy nên không dùng Sigmoid ở đầu ra)
        layers.append(nn.Linear(H, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
def build_model(H=10):
    model = RegressionMLP(H=H).to(device)

    # In bảng mô phỏng Keras Summary
    print(f'Model: "model_pytorch"\n')
    print(f"{'Layer (type)':<25} {'Output Shape':<20} {'Param #'}")
    print("=" * 60)
    print(f"{'Input (InputLayer)':<25} {str((None, 2)):<20} {0}")
    print("-" * 60)

    total_params = 0
    # Duyệt qua tất cả các module con kiểu nn.Linear
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            params = sum(p.numel() for p in module.parameters())
            total_params += params
            out_features = module.out_features
            # Phân biệt tên hiển thị cho giống Keras
            layer_name = f"{name} (Dense)" if name else "dense (Dense)"
            print(f"{layer_name:<25} {str((None, out_features)):<20} {params}")

    print("=" * 60)
    print(f"Total params: {total_params}")
    print(f"Trainable params: {total_params}")
    print(f"Non-trainable params: 0")
    print("-" * 60)

    return model

# Chạy lại kiểm tra
sample_model = build_model(H=10)

In [ ]:
# Số neuron trong từng lớp theo mô hình cơ sở
layer_sizes = [2, 10, 1]

layer_names = [
    "Input Layer\n(2 neurons)",
    "Hidden Layer\n(10 neurons)",
    "Output Layer\n(1 neuron)"
]

# Tọa độ các lớp
layer_x = [0.1, 0.5, 0.9]

positions = []

for i, size in enumerate(layer_sizes):
    y_positions = np.linspace(0.85, 0.15, size) if size > 1 else [0.5]
    positions.append([(layer_x[i], y) for y in y_positions])

# Tạo hình
fig, ax = plt.subplots(figsize=(12, 8))

# Vẽ các đường kết nối
for i in range(len(layer_sizes) - 1):
    for x1, y1 in positions[i]:
        for x2, y2 in positions[i + 1]:
            ax.plot(
                [x1, x2],
                [y1, y2],
                color="gray",
                linewidth=0.8,
                alpha=0.4,
                zorder=1
            )

# Màu sắc các lớp
colors = ["#4C91D8", "#43A88B", "#E5A044"]

# Nhãn neuron
labels = [
    ["x1", "x2"],
    [f"h{i+1}" for i in range(10)],
    ["y"]
]

# Vẽ neuron
for i, layer in enumerate(positions):
    for j, (x, y) in enumerate(layer):
        neuron = Circle(
            (x, y),
            0.025,
            facecolor=colors[i],
            edgecolor="black",
            linewidth=1,
            zorder=2
        )
        ax.add_patch(neuron)

        ax.text(
            x, y,
            labels[i][j],
            ha="center",
            va="center",
            fontsize=8,
            color="white",
            fontweight="bold",
            zorder=3
        )

# Tên các lớp
for i, x in enumerate(layer_x):
    ax.text(
        x, 0.97,
        layer_names[i],
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

# Hàm kích hoạt
ax.text(
    0.5, 0.05,
    "Hidden Activation: Sigmoid | Output Activation: Linear",
    ha="center",
    fontsize=10
)

# Tiêu đề
ax.set_title(
    "Neural Network Architecture (MLP)",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.08)
ax.set_aspect('equal', adjustable='box')  # Giữ neuron tròn, không bị méo
ax.axis("off")

plt.gcf().set_size_inches(14, 8)
plt.tight_layout()
save_figure("neural_network_architecture.png")
plt.show()

**Nhận xét về cấu trúc mạng MLP (Network Structure)**

- Mô hình MLP gồm 2 đầu vào, 1 tầng ẩn với 10 nơ-ron và 1 đầu ra. Hai đầu vào $(x_1)$, $(x_2)$ cung cấp tọa độ để mạng dự đoán giá trị $Gaussian$ ($y$).

- Tại tầng ẩn, mỗi nơ-ron thực hiện phép nhân đầu vào với trọng số, cộng bias và đưa kết quả qua hàm kích hoạt $Sigmoid$. Hàm này tạo tính phi tuyến, giúp mạng học được mối quan hệ phức tạp giữa đầu vào và đầu ra.

- Việc chọn $10 nơ-ron$ ẩn nhằm cung cấp đủ khả năng học các đặc trưng của hàm $Gaussian$, đồng thời giữ kiến trúc không quá phức tạp. Đây cũng là cấu hình cơ sở để khảo sát số lượng nơ-ron trong Experiment 2.

- Tầng đầu ra gồm $1 nơ-ron$ $Linear$, có nhiệm vụ kết hợp các đặc trưng đã học để tạo ra giá trị dự đoán ($y$). Thiết kế này phù hợp với bài toán hồi quy vì đầu ra là một giá trị liên tục.

- Tổng số tham số có thể huấn luyện là $41$, bao gồm trọng số và bias. Các tham số này được cập nhật trong quá trình huấn luyện nhằm giảm sai số giữa giá trị dự đoán và giá trị thực tế.

- Nhìn chung, kiến trúc này được sử dụng làm mô hình cơ sở để đánh giá ảnh hưởng của số lượng nơ-ron và số tầng ẩn đến khả năng xấp xỉ hàm $Gaussian$ hai chiều.

## EXPERIMENT 1

### Train Data

In [ ]:
# Khảo sát các quy mô tập train A theo yêu cầu đề bài
train_sizes = [200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000]

print(f"Các giá trị quy mô dữ liệu A sẽ khảo sát: {train_sizes}")

**Nhận xét:**
- Thí nghiệm khảo sát 10 quy mô dữ liệu huấn luyện, từ 200 đến 2000 mẫu, với bước tăng 200 mẫu mỗi lần.
- Các cấu hình sử dụng cùng kiến trúc $MLP$ với 10 nơ-ron ẩn để tập trung đánh giá ảnh hưởng của số lượng dữ liệu.
- Tập $validation$ và $test$ được giữ cố định nhằm so sánh kết quả giữa các quy mô train.

### Model & Training

In [ ]:
results_exp1 = []
criterion = nn.MSELoss()

for seed in SEEDS:
    for idx, size in enumerate(train_sizes):
        set_seed(seed)

        start_time = time.time()
        train_id = idx + 1

        # 1. Lấy dữ liệu train tương ứng với quy mô size A
        x_sub_train = x_train_norm[:size]
        y_sub_train = y_train[:size]

        train_ds = TensorDataset(
            torch.tensor(x_sub_train),
            torch.tensor(y_sub_train).unsqueeze(1)
        )
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

        valid_ds = TensorDataset(
            torch.tensor(x_valid_norm),
            torch.tensor(y_valid).unsqueeze(1)
        )
        valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False)

        # 2. Xây dựng mô hình với H=10
        print(f"Seed: {seed} | Train Size A = {size} (Train ID: {train_id})")
        model = build_model(H=10)

        optimizer = optim.SGD(model.parameters(), lr=0.01)
        epochs = 200

        history = {
            'loss': [], 'val_loss': [],
            'mean_squared_error': [], 'val_mean_squared_error': []
        }

        best_val_loss = float('inf')
        best_weights = None

        # 3. Training Loop
        for epoch in range(epochs):
            model.train()
            train_loss_sum = 0.0

            for inputs, targets in train_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()
                train_loss_sum += loss.item() * inputs.size(0)

            epoch_train_loss = train_loss_sum / len(train_loader.dataset)

            # Validation
            model.eval()
            val_loss_sum = 0.0

            with torch.no_grad():
                for inputs, targets in valid_loader:
                    inputs, targets = inputs.to(device), targets.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
                    val_loss_sum += loss.item() * inputs.size(0)

            epoch_val_loss = val_loss_sum / len(valid_loader.dataset)

            history['loss'].append(epoch_train_loss)
            history['val_loss'].append(epoch_val_loss)
            history['mean_squared_error'].append(epoch_train_loss)
            history['val_mean_squared_error'].append(epoch_val_loss)

            if epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                best_weights = copy.deepcopy(model.state_dict())

        training_time = (time.time() - start_time) / 60.0

        print(f"> Training time: {training_time:.4f} minutes")
        print(f"Train ID: {train_id}")

        # Khôi phục trọng số tốt nhất
        model.load_state_dict(best_weights)

        # 4. Lưu kết quả
        results_exp1.append({
            'seed': seed,
            'size': size,
            'model': model,
            'best_weights': best_weights,
            'history': history,
            'val_loss': best_val_loss,
            'time': training_time
        })

        # Lưu checkpoint riêng cho từng seed
        save_checkpoint(best_weights, f"exp1_train_size_{size}_seed_{seed}.pth", {
            "experiment": "Experiment 1",
            "train_size": size,
            "hidden_dim": 10,
            "seed": seed,
            "best_val_loss": float(best_val_loss)
        })

        log_message(
            f"EXP1 | seed={seed} | train_size={size} | "
            f"best_val_loss={best_val_loss:.8f} | "
            f"time_min={training_time:.4f}"
        )

In [ ]:
# Luu lich su huan luyen Experiment 1
save_history(results_exp1, "exp1")

log_message("Da luu lich su huan luyen Experiment 1.")

**Nhận xét:**
- Mỗi cấu hình được huấn luyện trong tối đa 200 $epoch$ bằng thuật toán $SGD$ với $learning$ $rate$ bằng $0.01$ và hàm mất mát $MSE$.
- Sau mỗi $epoch$, mô hình được đánh giá trên tập $validation$ và lưu lại trọng số khi $Validation$ $Loss$ đạt giá trị thấp nhất.
- Thời gian huấn luyện nhìn chung tăng khi quy mô dữ liệu lớn hơn, do mô hình phải xử lý nhiều mẫu hơn.

In [ ]:
# Lưu trọng số mô hình tốt nhất của Experiment 1
os.makedirs("checkpoints", exist_ok=True)

best_exp1 = min(results_exp1, key=lambda x: x["val_loss"])

torch.save({
    "model_state_dict": best_exp1["best_weights"],
    "train_size": best_exp1["size"],
    "hidden_dim": 10,
    "val_loss": best_exp1["val_loss"]
}, "checkpoints/exp1_best.pth")

print("Đã lưu mô hình tốt nhất của Experiment 1")
print(f"Train size: {best_exp1['size']}")
print(f"Validation Loss: {best_exp1['val_loss']:.6f}")
print("File: checkpoints/exp1_best.pth")


### Evaluate

In [ ]:
test_ds = TensorDataset(torch.tensor(x_test_norm), torch.tensor(y_test).unsqueeze(1))
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

exp1_summary = []

for res in results_exp1:
    model = res['model']
    model.load_state_dict(res['best_weights'])
    model.eval()

    test_loss_sum = 0.0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            test_loss_sum += loss.item() * inputs.size(0)

    test_mse = test_loss_sum / len(test_loader.dataset)
    test_rmse = np.sqrt(test_mse)

    exp1_summary.append({
        'A (Train Size)': res['size'],
        'Val Loss': res['val_loss'],
        'Test MSE': test_mse,
        'Test RMSE': test_rmse,
        'Time (min)': res['time']
    })

**Nhận xét:**
- Sai số thay đổi không hoàn toàn đồng đều khi quy mô dữ liệu tăng; cấu hình 1600 mẫu đạt kết quả tốt nhất với $Test$ $MSE$ = $0.002090$ và $Test$ $RMSE$ = $0.045712$.
- Ở quy mô 2000 mẫu, #Test# $MSE$ đạt $0.002100$, gần với kết quả tốt nhất nhưng không cải thiện thêm so với cấu hình 1600 mẫu.
- Kết quả cho thấy tăng dữ liệu có thể cải thiện khả năng dự đoán, tuy nhiên mức cải thiện còn phụ thuộc vào từng cấu hình và quá trình huấn luyện.

In [ ]:
df_exp1 = pd.DataFrame(exp1_summary)
print(df_exp1.to_string(index=False))

# Vẽ đồ thị biểu diễn Sai số (MSE/RMSE) theo Quy mô dữ liệu A
plt.figure(figsize=(10, 5))
plt.plot(df_exp1['A (Train Size)'], df_exp1['Test MSE'], marker='o', color='tab:red', linestyle='-', label='Test MSE')
plt.plot(df_exp1['A (Train Size)'], df_exp1['Test RMSE'], marker='s', color='tab:blue', linestyle='--', label='Test RMSE')
plt.title("Thí nghiệm 1: Đồ thị sai số theo quy mô tập dữ liệu huấn luyện (A)")
plt.xlabel("Quy mô dữ liệu huấn luyện (A)")
plt.ylabel("Giá trị sai số (Error)")
plt.xticks(train_sizes)
plt.legend()
plt.grid(True)
save_figure("exp1_error_by_train_size.png")
plt.show()

# Vẽ biểu đồ Loss & MSE song song cho mẫu đầu tiên (A = 200) để minh họa tiêu chuẩn
res_200 = results_exp1[0]
final_epoch = len(res_200['history']['loss']) - 1
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(res_200['history']['loss'], label='loss', color='tab:blue')
ax1.plot(res_200['history']['val_loss'], label='val_loss', color='tab:orange')
ax1.set_title(f"Loss at Epoch {final_epoch}: {res_200['history']['val_loss'][-1]:.6f}")
ax1.set_xlabel('epoch')
ax1.legend()

ax2.plot(res_200['history']['mean_squared_error'], label='mean_squared_error', color='tab:blue')
ax2.plot(res_200['history']['val_mean_squared_error'], label='val_mean_squared_error', color='tab:orange')
ax2.set_title(f"MSE at Epoch {final_epoch}: {res_200['history']['val_mean_squared_error'][-1]:.6f}")
ax2.set_xlabel('epoch')
ax2.legend()

plt.tight_layout()
save_figure("exp1_loss_history_train_size_200.png")
plt.show()
# Luu bang ket qua Experiment 1
df_exp1.to_csv(
    LOG_DIR / "exp1_results.csv",
    index=False,
    encoding="utf-8-sig"
)

log_message("Da luu bang ket qua Experiment 1.")

**Nhận xét:**
- Biểu đồ $Loss$ thể hiện sự thay đổi của sai số trên tập $train$ và $validation$ qua các $epoch$ trong quá trình huấn luyện.
- Hai đường $train$ và $validation$ giúp theo dõi khả năng học của mô hình cũng như phát hiện dấu hiệu sai lệch giữa kết quả trên hai tập dữ liệu.
- Biểu đồ $MSE$ thể hiện cùng xu hướng sai số dưới dạng chỉ số bình phương trung bình; cần đối chiếu hai đường để đánh giá quá trình huấn luyện.

In [ ]:
df_exp1 = pd.DataFrame(exp1_summary)
display(df_exp1)